In [3]:
from garmin_fit_sdk import Decoder, Stream

stream = Stream.from_file("24339076395_ACTIVITY.fit")
decoder = Decoder(stream)

messages, errors = decoder.read()

print(messages.keys())

dict_keys(['file_id_mesgs', 'file_creator_mesgs', 'activity_mesgs', '140', 'session_mesgs', 'time_in_zone_mesgs', 'lap_mesgs', '288', '327', '326', 'timestamp_correlation_mesgs', 'event_mesgs', 'device_info_mesgs', '22', '141', '394', 'device_settings_mesgs', 'user_profile_mesgs', '79', 'sport_mesgs', 'training_settings_mesgs', 'zones_target_mesgs', 'record_mesgs', '233', '325', '499', 'battery_mesgs'])


In [4]:

from garmin_fit_sdk import Decoder, Stream

# Load your Garmin FIT file
file_path = "24339076395_ACTIVITY.fit"

stream = Stream.from_file(file_path)
decoder = Decoder(stream)

messages, errors = decoder.read()

# Display decoding errors
if errors:
    print("Decoding errors:", errors)

print("\n========== FIT FILE CONTENTS ==========\n")

for message_type, records in messages.items():

    print(f"\nMESSAGE TYPE: {message_type}")
    print(f"NUMBER OF RECORDS: {len(records)}")

    if records:
        print("AVAILABLE FIELDS:")

        all_fields = set()

        for record in records:
            all_fields.update(record.keys())

        for field in sorted(all_fields):
            print(f"  - {field}")

print("\n=======================================")



========== FIT FILE CONTENTS ==========


MESSAGE TYPE: file_id_mesgs
NUMBER OF RECORDS: 1
AVAILABLE FIELDS:
  - garmin_product
  - manufacturer
  - product
  - serial_number
  - time_created
  - type

MESSAGE TYPE: file_creator_mesgs
NUMBER OF RECORDS: 1
AVAILABLE FIELDS:
  - software_version

MESSAGE TYPE: activity_mesgs
NUMBER OF RECORDS: 1
AVAILABLE FIELDS:
  - event
  - event_type
  - local_timestamp
  - num_sessions
  - timestamp
  - total_timer_time
  - type

MESSAGE TYPE: 140
NUMBER OF RECORDS: 1
AVAILABLE FIELDS:
  - 0
  - 1
  - 2
  - 3
  - 4
  - 5
  - 6
  - 7
  - 8
  - 9
  - 10
  - 11
  - 12
  - 13
  - 20
  - 22
  - 23
  - 24
  - 25
  - 26
  - 27
  - 28
  - 29
  - 30
  - 31
  - 32
  - 33
  - 34
  - 35
  - 36
  - 37
  - 38
  - 39
  - 40
  - 41
  - 43
  - 44
  - 45
  - 46
  - 48
  - 51
  - 52
  - 53
  - 54
  - 55
  - 56
  - 57
  - 58
  - 59
  - 60
  - 61
  - 63
  - 64
  - 65
  - 66
  - 67
  - 253

MESSAGE TYPE: session_mesgs
NUMBER OF RECORDS: 1
AVAILABLE FIELDS:


TypeError: '<' not supported between instances of 'int' and 'str'

In [5]:
sessions = messages.get("session_mesgs", [])

if sessions:
    session = sessions[0]

    print("Sport:", session.get("sport"))
    print("Distance:", session.get("total_distance"), "meters")
    print("Duration:", session.get("total_timer_time"), "seconds")
    print("Average HR:", session.get("avg_heart_rate"), "bpm")
    print("Maximum HR:", session.get("max_heart_rate"), "bpm")
    print("Calories:", session.get("total_calories"))

Sport: hiking
Distance: 14199.88 meters
Duration: 27569.485 seconds
Average HR: 144 bpm
Maximum HR: 172 bpm
Calories: 3480


In [6]:
import pandas as pd

records = messages.get("record_mesgs", [])

df = pd.DataFrame(records)

print(df.head())
print(df.columns.tolist())

df.to_csv("garmin_workout.csv", index=False)

                  timestamp  distance  enhanced_speed  enhanced_altitude  \
0 2026-09-12 12:32:14+00:00      0.00           0.000             3432.6   
1 2026-09-12 12:32:15+00:00      0.00           0.000             3432.8   
2 2026-09-12 12:32:24+00:00      0.97           0.970             3433.4   
3 2026-09-12 12:32:26+00:00      2.95           0.989             3433.6   
4 2026-09-12 12:32:30+00:00      5.08           1.017             3434.6   

   cycle_length16  heart_rate  cadence  temperature  fractional_cadence  107  \
0             0.0          98        0           28                 0.0    0   
1             0.0          98        0           28                 0.0    0   
2             0.0         101       51           28                 0.5    0   
3             0.0         103       51           28                 0.5    0   
4             0.0         107       48           28                 0.0    1   

   135  136  143  position_lat  position_long  
0  126   98   

In [7]:

from garmin_fit_sdk import Decoder, Stream
import pandas as pd

# Decode Garmin activity
stream = Stream.from_file("24339076395_ACTIVITY.fit")
decoder = Decoder(stream)

messages, errors = decoder.read()

# Extract records
records = messages.get("record_mesgs", [])
df = pd.DataFrame(records)

# Keep heart rate measurements
df = df[["timestamp", "heart_rate"]].dropna().copy()

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

# Example maximum heart rate (replace with your own)
MAX_HR = 190

# Heart rate zone boundaries
def get_hr_zone(hr):
    percentage = (hr / MAX_HR) * 100

    if percentage < 60:
        return "Zone 1"
    elif percentage < 70:
        return "Zone 2"
    elif percentage < 80:
        return "Zone 3"
    elif percentage < 90:
        return "Zone 4"
    else:
        return "Zone 5"

df["zone"] = df["heart_rate"].apply(get_hr_zone)

# Estimate time between measurements
df["duration"] = (
    df["timestamp"].shift(-1) - df["timestamp"]
).dt.total_seconds()

# Cap gaps to avoid counting long pauses
df["duration"] = df["duration"].clip(lower=0, upper=5).fillna(0)

# Calculate time spent in each zone
zone_time = df.groupby("zone")["duration"].sum() / 60

print("\nHeart Rate Zone Analysis")
print(zone_time.round(2))



Heart Rate Zone Analysis
zone
Zone 1      7.78
Zone 2     54.20
Zone 3    175.73
Zone 4    126.45
Zone 5      0.75
Name: duration, dtype: float64
